# Experiment 15: Institution-Aware Merging

**Problem diagnosed in nb43**: Naïve merging drops F1 by ~7.7 pp (LightGBM: 54.89% vs AUB-only 62.55%).
Two root causes to fix:

1. **Global citation threshold** — a single merged 75th-percentile mislabels papers because citation
   cultures differ across institutions. AUB papers that are locally high-impact may sit below the
   global bar, and vice-versa for peer papers.
2. **Institution identity ignored** — the model has no signal about *which* institution produced
   a paper, so it cannot learn institution-specific patterns.

**Configs tested** (LightGBM throughout — best model in nb43):

| Config | Train window | Threshold strategy | Institution feature |
|--------|-------------|-------------------|---------------------|
| Reference (nb43) | 2015-2017 | Global merged 75th pct | ✗ |
| A | 2015-2017 | Per-institution 75th pct | ✗ |
| B | 2010-2017 | Per-institution 75th pct | ✗ |
| C | 2010-2017 | Per-institution 75th pct | One-hot dummy |
| D | 2010-2017 | Per-institution 75th pct | Dummy + aggregate stats |

**Evaluation**: AUB test set 2018-2020 (same as nb43 Scenario A), plus per-institution breakdown.

In [ ]:
import sys
sys.path.append('../../')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
AUB_BASELINE_F1 = 0.6255   # nb43 AUB-only LR baseline
NB43_BEST_F1    = 0.5489   # nb43 LightGBM merged (global threshold, 2015-2017)
NB42_BEST_F1    = 0.6333   # nb42 domain segmentation best (AUB-only)

print('Libraries loaded')

## 1. Load Data

In [ ]:
data_path = Path('../../data/processed/all_unis_cleaned.pkl')
df = pd.read_pickle(data_path)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"\nPapers per institution:")
print(df['institution'].value_counts().to_string())
print(f"\nYear range: {df['Year'].min()} – {df['Year'].max()}")

## 2. Temporal Splits

Two training windows are tested:
- **Short** (2015-2017): same as nb43 for direct comparison
- **Expanded** (2010-2017): matches the best AUB-only split from nb42

In [ ]:
TEST_YEARS = [2018, 2019, 2020]

df_test = df[df['Year'].isin(TEST_YEARS)].copy()
df_test_aub = df_test[df_test['institution'] == 'AUB'].copy()

# Short window (matches nb43)
df_train_short = df[df['Year'].isin([2015, 2016, 2017])].copy()

# Expanded window (matches nb42 AUB-only best)
df_train_long = df[df['Year'].isin(range(2010, 2018))].copy()

print("SHORT train (2015-2017):")
print(df_train_short['institution'].value_counts().to_string())
print(f"  Total: {len(df_train_short):,}")

print("\nEXPANDED train (2010-2017):")
print(df_train_long['institution'].value_counts().to_string())
print(f"  Total: {len(df_train_long):,}")

print(f"\nTest – all unis (2018-2020): {len(df_test):,}")
print(f"Test – AUB only (2018-2020): {len(df_test_aub):,}")

## 3. Target Labeling Strategies

**Global** (nb43 baseline): single 75th-percentile over the entire training set.

**Per-institution**: each institution's papers are labeled relative to *their own* 75th percentile
citation count. This respects that Lehigh papers in Engineering may cite differently than AUB papers
in Medicine. The positive-class rate within each institution is kept at ~25%.

In [ ]:
def make_labels_global(df_tr, df_te_aub, df_te_all):
    """Single merged-75th-pct threshold (nb43 approach)."""
    thr = df_tr['Citations'].quantile(0.75)
    aub_thr = df_te_aub['Citations'].quantile(0.75)
    y_train  = (df_tr['Citations']     >= thr).astype(int)
    y_te_aub = (df_te_aub['Citations'] >= aub_thr).astype(int)
    y_te_all = (df_te_all['Citations'] >= thr).astype(int)
    print(f"  Global threshold:       {thr:.1f}  |  train pos rate: {y_train.mean():.1%}")
    return y_train, y_te_aub, y_te_all


def make_labels_per_institution(df_tr, df_te_aub, df_te_all):
    """Per-institution 75th-pct threshold.
    Train labels: each institution's own threshold.
    Test (AUB): AUB threshold from training data (no leakage).
    Test (all-unis): each institution's own threshold from training data.
    """
    y_train = pd.Series(0, index=df_tr.index)
    thresholds = {}
    for inst in df_tr['institution'].unique():
        mask = df_tr['institution'] == inst
        thr  = df_tr.loc[mask, 'Citations'].quantile(0.75)
        thresholds[inst] = thr
        y_train.loc[mask] = (df_tr.loc[mask, 'Citations'] >= thr).astype(int)
        print(f"  {inst:12s}: threshold={thr:6.1f}  pos_rate={y_train.loc[mask].mean():.1%}  n={mask.sum():,}")

    aub_thr  = thresholds.get('AUB', df_te_aub['Citations'].quantile(0.75))
    y_te_aub = (df_te_aub['Citations'] >= aub_thr).astype(int)

    y_te_all = pd.Series(0, index=df_te_all.index)
    for inst in df_te_all['institution'].unique():
        mask = df_te_all['institution'] == inst
        thr  = thresholds.get(inst, df_te_all.loc[mask, 'Citations'].quantile(0.75))
        y_te_all.loc[mask] = (df_te_all.loc[mask, 'Citations'] >= thr).astype(int)

    print(f"  Overall train pos rate: {y_train.mean():.1%}")
    return y_train, y_te_aub, y_te_all, thresholds


print("=== SHORT WINDOW — Global threshold ===")
y_short_global, y_te_aub_global, _ = make_labels_global(df_train_short, df_test_aub, df_test)

print("\n=== SHORT WINDOW — Per-institution thresholds ===")
y_short_inst, y_te_aub_inst, y_te_all_short_inst, thr_short = make_labels_per_institution(
    df_train_short, df_test_aub, df_test)

print("\n=== EXPANDED WINDOW — Per-institution thresholds ===")
y_long_inst, y_te_aub_long, y_te_all_long_inst, thr_long = make_labels_per_institution(
    df_train_long, df_test_aub, df_test)

## 4. Feature Engineering

Reusing the same feature pipeline as nb43. Institution features are added as an optional block.

In [ ]:
def preprocess_text(text):
    return str(text).lower() if pd.notna(text) else ""

def build_tfidf(df_tr, df_te_aub, df_te_all):
    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                            min_df=5, max_df=0.8, stop_words='english')
    def to_df(mat, idx):
        feat_names = [f'tfidf_{f}' for f in tfidf.get_feature_names_out()]
        return pd.DataFrame(mat.toarray(), index=idx, columns=feat_names)
    tr  = to_df(tfidf.fit_transform(df_tr['Abstract'].apply(preprocess_text)),  df_tr.index)
    tea = to_df(tfidf.transform(df_te_aub['Abstract'].apply(preprocess_text)),  df_te_aub.index)
    teo = to_df(tfidf.transform(df_te_all['Abstract'].apply(preprocess_text)),  df_te_all.index)
    return tr, tea, teo, tfidf


def build_venue_features(subset_df):
    def safe(s): return pd.to_numeric(s, errors='coerce')
    vf = pd.DataFrame(index=subset_df.index)
    col_map = {
        'snip': 'SNIP (publication year)',
        'snip_percentile': 'SNIP percentile (publication year) *',
        'citescore': 'CiteScore (publication year)',
        'citescore_percentile': 'CiteScore percentile (publication year) *',
        'sjr': 'SJR (publication year)',
        'sjr_percentile': 'SJR percentile (publication year) *',
    }
    available = {}
    for feat, col in col_map.items():
        if col in subset_df.columns:
            vf[feat] = safe(subset_df[col])
            available[feat] = col
    if available:
        pct_cols = [k for k in available if 'percentile' in k]
        if pct_cols:
            vf['avg_venue_percentile'] = vf[pct_cols].mean(axis=1)
            vf['is_top_journal'] = (vf['avg_venue_percentile'] >= 75).astype(int)
        else:
            raw_cols = [k for k in available if 'percentile' not in k]
            if raw_cols:
                vf['avg_venue_percentile'] = vf[raw_cols].mean(axis=1)
                vf['is_top_journal'] = (vf['avg_venue_percentile'] >= vf['avg_venue_percentile'].quantile(0.75)).astype(int)
    for col in vf.columns:
        vf[col] = vf[col].fillna(vf[col].median())
    return vf


def build_author_features(subset_df):
    af = pd.DataFrame(index=subset_df.index)
    for feat, col in [('num_authors', 'Number of Authors'),
                      ('num_institutions', 'Number of Institutions'),
                      ('num_countries', 'Number of Countries/Regions')]:
        af[feat] = pd.to_numeric(subset_df.get(col, pd.Series(np.nan, index=subset_df.index)), errors='coerce')
    af['is_single_author']        = (af['num_authors'] == 1).astype(int)
    af['is_international_collab'] = (af['num_countries'] > 1).astype(int)
    af['is_multi_institution']    = (af['num_institutions'] > 1).astype(int)
    af['authors_per_institution'] = (af['num_authors'] / af['num_institutions'].replace(0, np.nan)).fillna(1)
    for col in af.columns:
        af[col] = af[col].fillna(af[col].median())
    return af


def build_metadata_features(subset_df, pub_type_cols=None, source_type_cols=None):
    mf = pd.DataFrame(index=subset_df.index)
    mf['is_open_access']  = subset_df.get('Open Access', pd.Series(np.nan, index=subset_df.index)).notna().astype(int)
    mf['topic_prominence'] = pd.to_numeric(
        subset_df.get('Topic Prominence Percentile', pd.Series(np.nan, index=subset_df.index)), errors='coerce')
    mf['topic_prominence'] = mf['topic_prominence'].fillna(mf['topic_prominence'].median())

    pt = pd.get_dummies(subset_df.get('Publication Type', pd.Series('Unknown', index=subset_df.index)),
                        prefix='pub_type')
    if pub_type_cols is not None:
        pt = pt.reindex(columns=pub_type_cols, fill_value=0)

    st = pd.get_dummies(subset_df.get('Source Type', pd.Series('Unknown', index=subset_df.index)),
                        prefix='src_type')
    if source_type_cols is not None:
        st = st.reindex(columns=source_type_cols, fill_value=0)

    return pd.concat([mf, pt, st], axis=1), list(pt.columns), list(st.columns)


def build_interaction_features(venue_df, author_df):
    ix = pd.DataFrame(index=venue_df.index)
    if 'is_top_journal' in venue_df.columns and 'is_international_collab' in author_df.columns:
        ix['top_journal_x_intl_collab']    = venue_df['is_top_journal'] * author_df['is_international_collab']
    if 'avg_venue_percentile' in venue_df.columns:
        ix['venue_pct_x_num_authors']      = venue_df['avg_venue_percentile'] * author_df['num_authors']
        ix['venue_pct_x_num_institutions'] = venue_df['avg_venue_percentile'] * author_df['num_institutions']
    return ix


def build_features(df_tr, df_te_aub, df_te_all, add_institution=False, inst_stats_from=None):
    """Full feature pipeline. Returns (X_train, X_te_aub, X_te_all, tfidf)."""
    text_tr, text_tea, text_teo, tfidf = build_tfidf(df_tr, df_te_aub, df_te_all)

    v_tr  = build_venue_features(df_tr)
    v_tea = build_venue_features(df_te_aub)
    v_teo = build_venue_features(df_te_all)

    a_tr  = build_author_features(df_tr)
    a_tea = build_author_features(df_te_aub)
    a_teo = build_author_features(df_te_all)

    m_tr,  pt_cols, st_cols = build_metadata_features(df_tr)
    m_tea, _, _              = build_metadata_features(df_te_aub, pt_cols, st_cols)
    m_teo, _, _              = build_metadata_features(df_te_all, pt_cols, st_cols)

    i_tr  = build_interaction_features(v_tr,  a_tr)
    i_tea = build_interaction_features(v_tea, a_tea)
    i_teo = build_interaction_features(v_teo, a_teo)

    parts_tr  = [text_tr,  v_tr,  a_tr,  m_tr,  i_tr]
    parts_tea = [text_tea, v_tea, a_tea, m_tea, i_tea]
    parts_teo = [text_teo, v_teo, a_teo, m_teo, i_teo]

    if add_institution:
        # One-hot institution dummy
        inst_dummies_tr  = pd.get_dummies(df_tr['institution'],  prefix='inst')
        inst_cols        = inst_dummies_tr.columns.tolist()
        inst_dummies_tea = pd.get_dummies(df_te_aub['institution'], prefix='inst').reindex(columns=inst_cols, fill_value=0)
        inst_dummies_teo = pd.get_dummies(df_te_all['institution'], prefix='inst').reindex(columns=inst_cols, fill_value=0)
        parts_tr.append(inst_dummies_tr)
        parts_tea.append(inst_dummies_tea)
        parts_teo.append(inst_dummies_teo)

    if inst_stats_from is not None:
        # Institution-level aggregate stats (computed from training set only — no leakage)
        # Features: institution-level avg citation, median citation, high-impact rate
        inst_stats = (inst_stats_from.groupby('institution')['Citations']
                      .agg(inst_avg_cit='mean', inst_med_cit='median')
                      .reset_index())

        def add_inst_stats(subset_df):
            merged = subset_df[['institution']].merge(inst_stats, on='institution', how='left')
            merged.index = subset_df.index
            return merged[['inst_avg_cit', 'inst_med_cit']].fillna(merged[['inst_avg_cit', 'inst_med_cit']].median())

        parts_tr.append(add_inst_stats(df_tr))
        parts_tea.append(add_inst_stats(df_te_aub))
        parts_teo.append(add_inst_stats(df_te_all))

    X_tr  = pd.concat(parts_tr,  axis=1).fillna(0)
    X_tea = pd.concat(parts_tea, axis=1).fillna(0)
    X_teo = pd.concat(parts_teo, axis=1).fillna(0)
    print(f"  Feature matrix: train={X_tr.shape}  test_aub={X_tea.shape}  test_all={X_teo.shape}")
    return X_tr, X_tea, X_teo, tfidf


print("Feature builders defined.")

## 5. Evaluation Helper

In [ ]:
def evaluate(model, X_train, y_train, X_te_aub, y_te_aub, X_te_all, y_te_all,
             df_te_all, label=''):
    """Train, find optimal threshold on AUB test, return metrics dict."""
    model.fit(X_train, y_train)

    proba_aub = model.predict_proba(X_te_aub)[:, 1]
    proba_all = model.predict_proba(X_te_all)[:, 1]

    # Threshold optimisation on AUB test set
    thresholds = np.arange(0.10, 0.90, 0.01)
    f1s_aub = [f1_score(y_te_aub, (proba_aub >= t).astype(int), zero_division=0)
               for t in thresholds]
    best_thr = thresholds[int(np.argmax(f1s_aub))]
    y_pred_aub = (proba_aub >= best_thr).astype(int)

    aub_metrics = {
        'label':     label,
        'f1':        f1_score(y_te_aub, y_pred_aub, zero_division=0),
        'auc':       roc_auc_score(y_te_aub, proba_aub),
        'recall':    recall_score(y_te_aub, y_pred_aub, zero_division=0),
        'precision': precision_score(y_te_aub, y_pred_aub, zero_division=0),
        'threshold': best_thr,
        'n_train':   len(y_train),
    }

    # Per-institution breakdown using all-unis test
    inst_breakdown = {}
    for inst in sorted(df_te_all['institution'].unique()):
        mask = df_te_all['institution'] == inst
        if mask.sum() < 20:
            continue
        y_i = y_te_all[mask]
        p_i = proba_all[mask]
        f1s_i = [f1_score(y_i, (p_i >= t).astype(int), zero_division=0) for t in thresholds]
        t_i   = thresholds[int(np.argmax(f1s_i))]
        inst_breakdown[inst] = {
            'f1':    float(np.max(f1s_i)),
            'auc':   roc_auc_score(y_i, p_i) if len(np.unique(y_i)) > 1 else np.nan,
            'n':     int(mask.sum()),
            'thr':   t_i,
        }

    delta_vs_baseline  = (aub_metrics['f1'] - AUB_BASELINE_F1) * 100
    delta_vs_nb43_best = (aub_metrics['f1'] - NB43_BEST_F1) * 100
    marker = ' ← BEST' if aub_metrics['f1'] > NB43_BEST_F1 else ''
    print(f"{label:<55}  F1={aub_metrics['f1']*100:.2f}%  "
          f"AUC={aub_metrics['auc']*100:.2f}%  "
          f"Δbaseline={delta_vs_baseline:+.2f}pp  "
          f"Δnb43={delta_vs_nb43_best:+.2f}pp{marker}")

    return aub_metrics, inst_breakdown


results      = {}   # label -> aub_metrics
breakdowns   = {}   # label -> inst_breakdown
print("Evaluation helper defined.")

## 6. Config A — Per-institution Thresholds, Short Window (2015-2017)

Direct comparison with nb43: same training window, only the labeling strategy changes.

In [ ]:
print("Building features for SHORT window...")
X_tr_s, X_tea_s, X_teo_s, _ = build_features(df_train_short, df_test_aub, df_test)

lgbm_a = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                         class_weight='balanced', random_state=RANDOM_STATE,
                         n_jobs=-1, verbose=-1)

print("\nConfig A: per-institution labels, 2015-2017 train")
r, b = evaluate(lgbm_a, X_tr_s, y_short_inst, X_tea_s, y_te_aub_inst, X_teo_s,
                y_te_all_short_inst, df_test,
                label='Config A: per-inst labels, 2015-2017 (LightGBM)')
results['A'] = r
breakdowns['A'] = b

## 7. Config B — Per-institution Thresholds, Expanded Window (2010-2017)

In [ ]:
print("Building features for EXPANDED window...")
X_tr_l, X_tea_l, X_teo_l, _ = build_features(df_train_long, df_test_aub, df_test)

lgbm_b = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                         class_weight='balanced', random_state=RANDOM_STATE,
                         n_jobs=-1, verbose=-1)

print("\nConfig B: per-institution labels, 2010-2017 train")
r, b = evaluate(lgbm_b, X_tr_l, y_long_inst, X_tea_l, y_te_aub_long, X_teo_l,
                y_te_all_long_inst, df_test,
                label='Config B: per-inst labels, 2010-2017 (LightGBM)')
results['B'] = r
breakdowns['B'] = b

## 8. Config C — Per-institution Thresholds + Institution Dummy, Expanded Window

In [ ]:
print("Building features for EXPANDED window + institution dummy...")
X_tr_lc, X_tea_lc, X_teo_lc, _ = build_features(
    df_train_long, df_test_aub, df_test, add_institution=True)

lgbm_c = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                         class_weight='balanced', random_state=RANDOM_STATE,
                         n_jobs=-1, verbose=-1)

print("\nConfig C: per-inst labels + institution dummy, 2010-2017 train")
r, b = evaluate(lgbm_c, X_tr_lc, y_long_inst, X_tea_lc, y_te_aub_long, X_teo_lc,
                y_te_all_long_inst, df_test,
                label='Config C: per-inst labels + inst dummy, 2010-2017 (LightGBM)')
results['C'] = r
breakdowns['C'] = b

## 9. Config D — Per-institution Thresholds + Dummy + Aggregate Stats, Expanded Window

In [ ]:
print("Building features for EXPANDED window + institution dummy + aggregate stats...")
X_tr_ld, X_tea_ld, X_teo_ld, _ = build_features(
    df_train_long, df_test_aub, df_test,
    add_institution=True, inst_stats_from=df_train_long)

lgbm_d = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                         class_weight='balanced', random_state=RANDOM_STATE,
                         n_jobs=-1, verbose=-1)

print("\nConfig D: per-inst labels + inst dummy + aggregate stats, 2010-2017 train")
r, b = evaluate(lgbm_d, X_tr_ld, y_long_inst, X_tea_ld, y_te_aub_long, X_teo_ld,
                y_te_all_long_inst, df_test,
                label='Config D: per-inst labels + inst feats, 2010-2017 (LightGBM)')
results['D'] = r
breakdowns['D'] = b

## 10. LR Comparison on Best Config

Repeat the best-performing config with LR for completeness.

In [ ]:
# We'll pick the best config after seeing results — placeholder runs best candidate (C or D)
# Adjust 'best_key' after running cells above
best_key = max(results, key=lambda k: results[k]['f1'])
print(f"Best config so far: {best_key}  F1={results[best_key]['f1']*100:.2f}%")

if best_key in ('C', 'D'):
    X_tr_best, X_tea_best, X_teo_best = (X_tr_ld, X_tea_ld, X_teo_ld) if best_key == 'D' else (X_tr_lc, X_tea_lc, X_teo_lc)
else:
    X_tr_best, X_tea_best, X_teo_best = X_tr_l, X_tea_l, X_teo_l

lr_best = LogisticRegression(max_iter=1000, class_weight='balanced',
                              n_jobs=-1, random_state=RANDOM_STATE)
r, b = evaluate(lr_best, X_tr_best, y_long_inst, X_tea_best, y_te_aub_long, X_teo_best,
                y_te_all_long_inst, df_test,
                label=f'Config {best_key} best (LR)')
results[f'{best_key}_lr'] = r
breakdowns[f'{best_key}_lr'] = b

## 11. Results Summary

In [ ]:
print("=" * 110)
print("EXPERIMENT 15 — INSTITUTION-AWARE MERGING RESULTS (AUB test set 2018-2020)")
print("=" * 110)
print(f"{'Config':<60}  {'F1':>7}  {'AUC':>7}  {'Recall':>7}  {'Prec':>7}  {'Δ baseline':>11}  {'Δ nb43':>9}")
print("-" * 110)

reference_row = {'label': 'Reference: nb43 LightGBM (global thr, 2015-2017)', 'f1': NB43_BEST_F1,
                 'auc': 0.8149, 'recall': 0.6226, 'precision': 0.4908}
for row in [reference_row] + list(results.values()):
    db = (row['f1'] - AUB_BASELINE_F1) * 100
    dn = (row['f1'] - NB43_BEST_F1) * 100
    mark = ' ← BEST' if row['f1'] == max(r['f1'] for r in results.values()) and row != reference_row else ''
    print(f"{row['label']:<60}  {row['f1']*100:>6.2f}%  {row['auc']*100:>6.2f}%  "
          f"{row['recall']*100:>6.2f}%  {row['precision']*100:>6.2f}%  "
          f"{db:>+10.2f}pp  {dn:>+8.2f}pp{mark}")

print("=" * 110)
print(f"AUB-only baseline (ref): {AUB_BASELINE_F1*100:.2f}%   nb42 best (ref): {NB42_BEST_F1*100:.2f}%")

## 12. Per-Institution Breakdown

How well does each config generalise across all four institutions?

In [ ]:
best_key_all = max(results, key=lambda k: results[k]['f1'])
bd = breakdowns[best_key_all]

print(f"Per-institution breakdown — {results[best_key_all]['label']}")
print("=" * 60)
print(f"{'Institution':<15}  {'F1':>7}  {'AUC':>7}  {'N test':>8}")
print("-" * 60)
for inst, m in sorted(bd.items(), key=lambda x: -x[1]['f1']):
    print(f"{inst:<15}  {m['f1']*100:>6.2f}%  {m['auc']*100:>6.2f}%  {m['n']:>7,}")
print("=" * 60)

# Compare all configs across institutions
print("\nF1 by institution — all configs:")
header = f"{'Institution':<15}" + "".join(f"  {k:>10}" for k in results)
print(header)
print("-" * len(header))
for inst in sorted(df_test['institution'].unique()):
    row_str = f"{inst:<15}"
    for k, bd in breakdowns.items():
        val = bd.get(inst, {}).get('f1', float('nan'))
        row_str += f"  {val*100:>9.2f}%" if not np.isnan(val) else f"  {'—':>9}"
    print(row_str)

## 13. Feature Importance — Institution Features

How much weight does LightGBM assign to the institution dummies and aggregate stats?

In [ ]:
# Use Config D (most features) or C
target_model = lgbm_d if 'D' in results else lgbm_c
target_X     = X_tr_ld if 'D' in results else X_tr_lc

importances = pd.Series(target_model.feature_importances_, index=target_X.columns)

# Filter to institution-related features
inst_feats = importances[[c for c in importances.index
                           if c.startswith('inst_') or c in ('inst_avg_cit', 'inst_med_cit')]]

print("Institution feature importances (gain):")
print(inst_feats.sort_values(ascending=False).to_string())

print("\nTop-20 features overall:")
print(importances.sort_values(ascending=False).head(20).to_string())

## 14. Save Best Model

In [ ]:
models_dir  = Path('../models')
metrics_dir = Path('../../reports/metrics')
models_dir.mkdir(exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

best_key_save = max(results, key=lambda k: results[k]['f1'])
best_models = {'C': lgbm_c, 'D': lgbm_d, 'B': lgbm_b, 'A': lgbm_a}
best_model_obj = best_models.get(best_key_save.split('_')[0])

if best_model_obj is not None:
    out = models_dir / f'exp15_inst_aware_{best_key_save.lower()}_lgbm.pkl'
    with open(out, 'wb') as f:
        pickle.dump(best_model_obj, f)
    print(f"Saved: {out}")

# Save metrics summary
summary_rows = [{'experiment': 'exp15', 'config': k, **v} for k, v in results.items()]
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(metrics_dir / 'exp15_institution_aware_merging.csv', index=False)
print(f"Saved metrics: {metrics_dir / 'exp15_institution_aware_merging.csv'}")